# model_14 - YOL MODELI

Mimari `deneme2/DENKLEM.md`, olcu `olcme_14`:

    DOGRULUK
      BICIM   kalip / ek / tip        cumle OGRETILDIGI GIBI mi kuruldu
      BILGI   OGRETILEN / CIKARIM     cumledeki bilgi dogru mu

**Kural 8 - butun kosular ARKA PLANDA.** Uzun is bir iplikte baslar,
hucre ANINDA doner. Durum icin `S DURUM` (kisa) ya da `R RAPOR` (tam):
CPU, aninda doner, her zaman guvenli.

**Korpus burada URETILMEZ.** Birim akisi Drive'dan okunur
(`MyDrive/model_14/birim_14.npz`, ~9 MB). Grafi `veri_kur` tohumdan
1 sn'de kuruyor. Karakter yolu (jeton, 512'lik paketleme, 164 MB
onbellek) Colab'da hic calismaz - model_14 karakter gormuyor.

**Tek tohum (t0).** Mimarinin ogrenilen kismi donmeler; sabit nokta
bulutu tohumdan geliyor ama donmeler hangi buluta verilirse ona uyum
sagliyor. Olumsuz sonucta t1/t2 eklenir (ISIMLENDIRME.md).

Hucre sirasi: 0 GPU kapisi / 1 depo+Drive / 2 veri+sinav /
3 egitim / X durdur / P profil / 4 dogruluk / S durum / R rapor /
5 nabiz.


In [ ]:
# 0 GPU KAPISI  |  GPU  |  tekrar: GUVENLI
# CLAUDE.md kural 2: GPU'yu kullanacak hucre GPU'yu KENDI ICINDE sorar.
# Ayri bir "GPU var mi" hucresi hucre sirasina bagli bir kuraldir --
# insan hatirlarsa calisir.
import os, sys, time, json, threading, subprocess
import torch

assert torch.cuda.is_available(), "GPU YOK -- Runtime > Change runtime type"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, "GPU'da sadece %.1f GB bos" % _bos
print("GPU kapisi GECTI: %s  bos %.1f GB" % (torch.cuda.get_device_name(0), _bos))
print("torch", torch.__version__)


In [ ]:
# 1 DEPO + DRIVE  |  CPU  |  tekrar: GUVENLI  |  ~15 sn
from google.colab import drive
drive.mount("/content/drive")

DEPO = "https://github.com/sekerahmet/sekerai.git"
KOD = "/content/kod"
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", DEPO, KOD], check=True)
KOL = KOD + "/deneme2/model_14"
if KOL not in sys.path:
    sys.path.insert(0, KOL)

# !! MODUL ONBELLEGI. rm -rf + clone dosyayi degistirir ama `import`
# ONCEKI modulu dondurur -- model_13'te bir kez bunun kurbani olundu.
for _m in [m for m in list(sys.modules) if m.endswith("_14")]:
    del sys.modules[_m]

CIKTI = "/content/drive/MyDrive/model_14/t0"
BIRIM_YOL = "/content/drive/MyDrive/model_14/birim_14.npz"
os.makedirs(CIKTI, exist_ok=True)

# =====================================================================
# CLAUDE.md KURAL 9 -- MEKANIK KAPI
# =====================================================================
# "Drive'dan alinabilecek her sey Drive'dan. Colab yalniz ZORUNLU
#  olani uretir."  Birim akisi Drive'dan GELIR; Colab onu URETMEZ.
# Dosya yoksa burasi DURUR -- sessizce 300 sn uretmeye baslamaz.
#
# Gerekce (20 Eylul, ayni gun IKI KEZ yasandi): uretmek her oturumda
# ~300 sn goturuyor, KARAKTER YOLUNU (jeton + 512'lik paketleme +
# 164 MB ara tensor) geri getiriyor -- model_14 karakter gormuyor --
# ve her kosuda yeniden uretilen korpus bolucudeki en ufak
# degisiklikte SESSIZCE kayiyor.
assert os.path.exists(BIRIM_YOL), (
    "BIRIM AKISI DRIVE'DA YOK: " + BIRIM_YOL + "
"
    "Colab korpus URETMEZ (CLAUDE.md kural 9). Yerelde uretip koyun:
"
    "    import ayar_14 as AY, birim_14 as BR
"
    "    BR.kur(AY.AYAR, onbellek=r\"G:/Drive'im/model_14/birim_14.npz\")")

COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
print("cikti ", CIKTI)
print("birim ", BIRIM_YOL, "%.1f MB" % (os.path.getsize(BIRIM_YOL) / 1e6))


In [ ]:
# 2 VERI + SINAV  |  CPU  |  ARKA PLAN  (CLAUDE.md kural 8)
#
#   graf         tohumdan uretilir       ~1 sn   (dosya gerekmez)
#   BIRIM AKISI  Drive'dan OKUNUR        ~4 sn
#
# !! `BR.yukle` -- `BR.kur` DEGIL. `kur`, dosya yoksa URETIR; burada
# uretmek YASAK (kural 9, hucre 1'deki kapi). Iz okunmuyor, akistan
# YENIDEN HESAPLANIP karsilastiriliyor: dosya bozuksa ya da baska bir
# bolucuyle uretilmisse burada patlar, sessizce gecmez.
KURULUM, KLOG = "kosuyor", []
T_KUR = [0.0, 0.0]


def _kur():
    global KURULUM
    T_KUR[0] = time.time()
    try:
        g = globals()
        import ayar_14 as AY, taban_14 as MT, veri_14 as V14
        import birim_14 as BR, olcme_14 as OL, model_14 as M14
        g.update(AY=AY, MT=MT, V14=V14, BR=BR, OL=OL, M14=M14)
        A = AY.AYAR

        v = MT.veri_kur(A, yaz=lambda *a, **k: None)
        L = MT.olcme_listeleri(A, v)
        assert V14.IZ == AY.IZ_GRAF, V14.IZ
        assert MT.olcme_izi(L) == AY.IZ_OLCME, MT.olcme_izi(L)
        KLOG.append("izler TUTUYOR  graf %s  olcme %s  (%.0f sn)"
                    % (V14.IZ, MT.olcme_izi(L), time.time() - T_KUR[0]))

        b = BR.yukle(BIRIM_YOL, yaz=KLOG.append)
        # !! PENCERE de denetleniyor: sinav cevabi sorunun OZNESINDEN
        # uretiyor, ikisi ayni zincire sigmazsa model o baglantiyi HIC
        # gormez. L=16 ciftlerin %77,2'sini kapsiyordu ve bu OLCULMEDEN
        # secilmisti; kapi bos degil, 16 ve 20 reddediliyor.
        KLOG.append(BR.kapi(b, AY.PENCERE))

        G = V14.kur(A.veri_tohum)
        E_ad = [x for t in V14.TIPLER for x in G["ad"][t]]
        g.update(v=v, L=L, b=b, E_ad=E_ad)
        KURULUM = "bitti"
    except Exception as e:
        import traceback
        KURULUM = "HATA: %s: %s" % (type(e).__name__, e)
        KLOG.append(traceback.format_exc()[-700:])
    T_KUR[1] = time.time()


threading.Thread(target=_kur, daemon=True).start()
print("kurulum ARKA PLANDA basladi -- durum icin S")


In [ ]:
# 3 EGITIM  |  GPU  |  ARKA PLAN
# !! Her adimda GRADYANIN sonlu olup olmadigina bakiyor, `op.step()`
# ONCESINDE. Gerekce OLCULDU: itme terimindeki sqrt tekilligi adim
# 1'in GRADYANINI NaN yapiyordu ve adim 2'de parametreler zaten bozuk
# oldugu icin orada bakmak GEC KALIYORDU.
# Bedeli: adim basina 5 kucuk reduksiyon + tek senkron (~0,3 ms);
# `float(uye)` zaten senkron yapiyordu.
EGITIM, ELOG, MDL = "kosuyor", [], None
EP = [0, 0]
T_EG = [0.0, 0.0]


def _egit():
    global EGITIM, MDL
    T_EG[0] = time.time()
    try:
        assert KURULUM == "bitti", "once 2 bitsin (durum: %s)" % KURULUM
        A = AY.AYAR
        torch.manual_seed(A.tohum)
        m = M14.Yol(len(b), D=AY.D_DURUM, d=AY.D_OKUMA, K=AY.K_KOD,
                    tam=M14.sinif_ayir(b.say, AY.K_TAM),
                    saat=AY.SAAT).to("cuda")
        ELOG.append(M14.kapi(m).replace("
", "
  "))

        P = torch.as_tensor(b.pencere(AY.PENCERE, AY.ATLA).copy(),
                            device="cuda")
        op = torch.optim.Adam(m.parameters(), lr=AY.LR)
        gg = torch.Generator(device="cuda").manual_seed(A.tohum)
        N, BS = len(P), AY.BATCH
        EP[1] = AY.EPOK
        ELOG.append("pencere %s  atla %d  adim/epok %d  batch %d"
                    % (tuple(P.shape), AY.ATLA, (N + BS - 1) // BS, BS))

        def dok(baslik, ad):
            ELOG.append("  !! %s -- adim %d" % (baslik, ad))
            ELOG.append("  " + M14.saglik(m).replace("
", "
  "))
            for k, p in m.named_parameters():
                g_ = p.grad
                iy = torch.isfinite(g_) if g_ is not None else None
                ELOG.append(
                    "     %-3s |p|max %9.3e sonlu %-5s | |g|max %9.3e"
                    "  SONSUZ %d/%d"
                    % (k, float(p.abs().max()), bool(torch.isfinite(p).all()),
                       float(g_[iy].abs().max()) if iy is not None
                       and bool(iy.any()) else float("nan"),
                       int((~iy).sum()) if iy is not None else -1,
                       p.numel()))

        ad = 0
        for ep in range(1, AY.EPOK + 1):
            perm = torch.randperm(N, device="cuda", generator=gg)
            tot = ns = 0.0
            for i in range(0, N, BS):
                ad += 1
                X = P[perm[i:i + BS]]
                k, uye = m.kayip(X, AY.A1_DIS, AY.A2_CAPA, AY.A3_DUZEN,
                                 AY.DELTA, AY.BETA, AY.R_CAPA)
                op.zero_grad(set_to_none=True)
                k.backward()
                # TEK senkron: butun gradyanlarin toplami sonlu mu.
                # inf ya da nan varsa toplam da sonlu olmaz.
                g = float(sum(p.grad.sum() for p in m.parameters()))
                kf = float(k)
                if ad <= 2 or g != g or abs(g) == float("inf")                         or kf != kf or abs(kf) == float("inf"):
                    bas = ("ILK ADIMLAR" if ad <= 2 and g == g
                           and abs(g) != float("inf") and kf == kf
                           else "SONLU DEGIL")
                    dok("%s  kayip %s  grad toplam %s" % (bas, kf, g), ad)
                    if bas == "SONLU DEGIL":
                        raise RuntimeError("adim %d: kayip %s grad %s"
                                           % (ad, kf, g))
                op.step()
                tot += float(uye) * len(X)
                ns += len(X)
            EP[0] = ep
            ELOG.append("  epok %d/%d  uye %.4f  (%.0f sn)"
                        % (ep, AY.EPOK, tot / ns, time.time() - T_EG[0]))
            ELOG.append("       " + M14.saglik(m).replace("
", "
       "))
        MDL = m
        EGITIM = "bitti"
    except Exception as e:
        import traceback
        EGITIM = "HATA: %s: %s" % (type(e).__name__, e)
        ELOG.append(traceback.format_exc()[-700:])
    T_EG[1] = time.time()


threading.Thread(target=_egit, daemon=True).start()
print("egitim ARKA PLANDA basladi -- durum icin S")


In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Egitim ipligi daemon; durdurma bayragi YOK. Cekirdegi oldurmek
# KURULUMU da goturur; onun yerine `kayip` gecici olarak hata firlatir
# yapiliyor -- iplik kendi `except`ine duser, kurulum (v/L/b/E_ad)
# ayakta kalir.
import gc

_asil = M14.Yol.kayip
M14.Yol.kayip = lambda *a, **k: (_ for _ in ()).throw(
    RuntimeError("DURDURULDU"))
for _ in range(120):
    if EGITIM != "kosuyor":
        break
    time.sleep(0.5)
M14.Yol.kayip = _asil          # geri tak
MDL = None
gc.collect()
torch.cuda.empty_cache()
_t, _b = torch.cuda.mem_get_info()
print("egitim :", EGITIM)
print("kurulum:", KURULUM)
print("GPU    : %.1f / %.1f GB" % ((_b - _t) / 1e9, _b / 1e9))
